# NetSense-QoR — training on Kaggle

**One-time setup**
1. On your laptop: `python -m src.make_subset --data-root OPENABC2_DATASET --out kaggle_data/openabc-compact` (44 files, about 80 MB).
2. On Kaggle: *Datasets → New Dataset*, upload the **contents** of `kaggle_data/openabc-compact` (the `graphs` folder, `recipes.json`, `synthesisStatistics.pickle`), name it `openabc-mini`.
3. Create a notebook, *Add Input* → your `openabc-mini` dataset, then in *Settings*: **Accelerator = GPU**, **Internet = on**.
4. Upload this notebook (or copy its cells) and *Run All*.

Outputs are written to `/kaggle/working/runs/` and appear under the notebook's *Output* tab.

In [ ]:
!git clone https://github.com/tejaswini-18/netsense.git
%cd netsense
!git log --oneline -1

In [ ]:
# Kaggle already has PyTorch (with GPU support), pandas, scipy, matplotlib, tqdm and pyyaml.
# Install only what is missing. Do NOT reinstall torch: that breaks the running session.
!pip install -q torch_geometric pytest
import torch, torch_geometric
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# Find the attached dataset (the folder that contains synthesisStatistics.pickle)
from pathlib import Path
DATA_ROOT = next(Path("/kaggle/input").rglob("synthesisStatistics.pickle")).parent
print("Dataset:", DATA_ROOT, "| design graphs:", len(list((DATA_ROOT / "graphs").glob("*.pt"))), "(expected 42)")

In [ ]:
# Quick check that everything works here before the long run
!NETSENSE_DATA_ROOT={DATA_ROOT} python -m pytest -q

In [ ]:
!python -m src.train --config configs/baseline.yaml --data-root {DATA_ROOT} --runs-dir /kaggle/working/runs

In [ ]:
# Show the results of the latest run
import json
from IPython.display import Image, display
run_dir = sorted(Path("/kaggle/working/runs").iterdir())[-1]
print(run_dir)
print(json.dumps(json.loads((run_dir / "test_metrics.json").read_text()), indent=2))
display(Image(str(run_dir / "loss_curve.png")))
display(Image(str(run_dir / "test_scatter.png")))